# Tutorial 4 — MOSAIC mode: β Pic b

**What you'll learn:** How to combine a medium-resolution spectrum and multi-band
photometry into a single simultaneous fit using ForMoSA's MOSAIC mode.

**Target:** β Pictoris b — the archetypal directly-imaged planet, at ~19.7 pc,
discovered in 2008. It has been observed by many instruments across a wide
wavelength range, making it an ideal MOSAIC benchmark.

**MOSAIC mode:** Each instrument/dataset gets its own likelihood. ForMoSA computes
a *meta-likelihood* (sum of individual log-likelihoods) and fits a per-instrument
analytical scaling factor (alpha_i) to account for flux calibration offsets between
different instruments.

**Parameters fitted:** Teff, log g, r, d, rv, alpha_0 (spectrum), alpha_1 (photometry)

> **Note:** The data for this tutorial is not yet publicly available.
> Data download cells are placeholders. All other code is complete.


## Section 0: Setup

In [ ]:
import sys
try:
    import ForMoSA
    print(f"ForMoSA {ForMoSA.__version__} — OK")
except ImportError:
    raise ImportError("pip install ForMoSA && conda install dask netCDF4 bottleneck")
print(f"Python {sys.version.split()[0]}")


In [ ]:
from pathlib import Path
TUTORIAL_DIR = Path(".").resolve()
for d in ["data", "adapted_grid", "results", "grid"]:
    (TUTORIAL_DIR / d).mkdir(exist_ok=True)
print(f"Working directory: {TUTORIAL_DIR}")


In [ ]:
# Cell C: Data download + validation
# TODO: Replace URLs once β Pic b MOSAIC dataset is published.
# MOSAIC requires TWO FITS files: one spectroscopic, one photometric.
#   Spec  (BetaPicb_spectrum.fits):  WAV, WAVE_UNIT, FLX, ERR, RES
#   Photo (BetaPicb_photo.fits):     WAV, WAVE_UNIT, FLX, ERR, FAC, INS, FILT

from astropy.io import fits

SPEC_FILE  = TUTORIAL_DIR / "data" / "BetaPicb_spectrum.fits"
PHOTO_FILE = TUTORIAL_DIR / "data" / "BetaPicb_photo.fits"

for label, fpath, required in [
    ("Spectrum",  SPEC_FILE,  {"WAV", "WAVE_UNIT", "FLX", "ERR", "RES"}),
    ("Photometry", PHOTO_FILE, {"WAV", "WAVE_UNIT", "FLX", "ERR", "FAC", "INS", "FILT"}),
]:
    if not fpath.exists():
        raise FileNotFoundError(
            f"{label} file not found: {fpath}\n"
            "β Pic b MOSAIC data is not yet publicly available. "
            "The download URL will be added here once the dataset is published."
        )
    with fits.open(fpath) as hdul:
        found = {ext.name.upper() for ext in hdul[1:]}
    missing = required - found
    if missing:
        raise RuntimeError(f"{label}: missing extensions {missing}")
    print(f"{label} ({fpath.name}): OK — extensions: {sorted(found)}")


In [ ]:
# Cell D: Grid download (BT-Settl — same as Tutorials 1 & 2)
import urllib.request

GRID_FILE = TUTORIAL_DIR / "grid" / "BT-Settl.nc"
GRID_URL  = (
    "https://github.com/exoAtmospheres/ForMoSA/releases/download/"
    "tutorial-data-v1/BT-Settl.nc"
)

if not GRID_FILE.exists():
    print("Downloading BT-Settl grid (~1 GB). Keep this file for all tutorials.")
    try:
        from tqdm import tqdm
        class _P(tqdm):
            def update_to(self, b=1, bs=1, ts=None):
                if ts: self.total = ts
                self.update(b * bs - self.n)
        with _P(unit="B", unit_scale=True, desc="BT-Settl.nc") as t:
            urllib.request.urlretrieve(GRID_URL, GRID_FILE, reporthook=t.update_to)
    except ImportError:
        def _p(c, bs, tot):
            print(f"\r  {min(100,c*bs/tot*100):.1f}%", end="", flush=True)
        urllib.request.urlretrieve(GRID_URL, GRID_FILE, reporthook=_p); print()
else:
    print(f"Grid already present: {GRID_FILE.name}")

import xarray as xr
ds = xr.open_dataset(GRID_FILE, decode_cf=False)
print(f"Grid: {dict(ds.sizes)}")


## Section 1: The science

### What is MOSAIC mode?

MOSAIC mode allows you to fit **multiple datasets simultaneously** with a single
nested sampling run. Each dataset gets its own likelihood:

```
log L_total = log L_spectrum + log L_photometry
```

ForMoSA also fits a per-dataset **analytical scaling factor** (alpha_i) that
accounts for intercalibration offsets between different instruments —
for example, a 10% offset between a ground-based spectrum and space photometry.
This makes the fit robust to absolute flux calibration uncertainties.

### When to use MOSAIC

Use MOSAIC when you have:
- Observations from different instruments with different resolutions
- Potential flux calibration offsets between datasets
- More data than a single observation provides

**Pitfall:** Too many alpha parameters with too few data points leads to
overfitting. Rule of thumb: you need at least ~10× more data points than
free alpha parameters.

### β Pic b

β Pictoris b is one of the best-characterised directly-imaged planets:
- Mass: ~13 MJup (from radial velocity and astrometry)
- Distance: 19.7 pc (Gaia)
- Age: ~23 Myr (β Pic moving group)
- Spectrum: L-type, strong CO absorption, measurable vsini (~25 km/s)


## Section 2: Inspect the data

In [ ]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np

with fits.open(SPEC_FILE) as hdul:
    wav_s = hdul["WAV"].data.astype(float)
    flx_s = hdul["FLX"].data.astype(float)
    err_s = hdul["ERR"].data.astype(float)

with fits.open(PHOTO_FILE) as hdul:
    wav_p = hdul["WAV"].data.astype(float)
    flx_p = hdul["FLX"].data.astype(float)
    err_p = hdul["ERR"].data.astype(float)

print(f"Spectrum : {len(wav_s)} points, {wav_s.min():.3f}–{wav_s.max():.3f} µm")
print(f"Photometry: {len(wav_p)} points, {wav_p.min():.3f}–{wav_p.max():.3f} µm")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(wav_s, flx_s, lw=0.7, color="#2E86AB", label="Spectrum (instrument 0)")
ax.errorbar(wav_p, flx_p, yerr=err_p, fmt="o", color="#E84855",
            capsize=4, label="Photometry (instrument 1)")
ax.set_xlabel(r"Wavelength (µm)")
ax.set_ylabel(r"Flux (W m$^{-2}$ µm$^{-1}$)")
ax.legend(); plt.tight_layout(); plt.show()


## Section 3: Configure the analysis

In [ ]:
from ForMoSA.config.global_config import ConfigPath, ConfigAdapt, ConfigInversion, ConfigParameters

# MOSAIC: provide both files in observation_path (order matters — index 0, 1, ...)
config_path = ConfigPath(
    observation_path=[str(SPEC_FILE), str(PHOTO_FILE)],
    adapt_store_path=str(TUTORIAL_DIR / "adapted_grid"),
    result_path=str(TUTORIAL_DIR / "results"),
    model_path=str(GRID_FILE),
)

# Two observations: list parameters per observation where they differ.
# Single-element lists are broadcast to all observations.
config_adapt = ConfigAdapt(
    res_cont=["500", "NA"],   # continuum removal for spectrum; none for photometry
)

config_inversion = ConfigInversion(
    wav_fit=["2.0, 2.45", "1.0, 5.0"],  # per-observation fitting windows
    ns_algo="nestle",
    npoints=300,
    logL_type=["chi2", "chi2"],          # one likelihood type per observation
)

config_params = ConfigParameters(
    par1=["uniform", "1200", "3000"],  # Teff (K)
    par2=["uniform", "2.5", "5.5"],   # log g (dex)
    r=["uniform", "0.5", "3.0"],      # radius (R_Jupiter)
    d=["constant", "19.7"],           # distance (pc) — Gaia
    rv=["uniform", "-100", "100"],    # radial velocity (km/s)
    # MOSAIC alpha parameters: alpha_0 for spectrum, alpha_1 for photometry
    # These account for intercalibration offsets between the two instruments.
    alpha=["uniform", "0.1", "5.0", "uniform", "0.1", "5.0"],
    # Syntax: [prior_0, min_0, max_0, prior_1, min_1, max_1, ...]
)

print("MOSAIC configuration:")
print(f"  Observations : spectrum (idx 0) + photometry (idx 1)")
print(f"  wav_fit      : {config_inversion.wav_fit}")
print(f"  Free params  : par1, par2, r, rv, alpha_0, alpha_1")
print(f"  Fixed        : d = 19.7 pc")


## Section 4: Adapt the grid

In [ ]:
from ForMoSA import Analysis

adapted = False
analysis = Analysis(config_path, adapted=adapted, fitted=False)
if not adapted:
    print("Adapting grid to 2 observations...")
    analysis.adapt(config_adapt, config_inversion)
    print("Done.")


## Section 5: Run the nested sampling fit

In [ ]:
from ForMoSA.config.global_config import Config_NS

config_ns = Config_NS()
print(f"Running {config_inversion.ns_algo} with {config_inversion.npoints} live points...")
analysis.nested_sampling(config_params, config_adapt, config_inversion, config_NS=config_ns)
print("Fit complete.")


## Section 6: Results

In [ ]:
analysis.plot(analysis.ns.results)
print(analysis.ns.results.summary(sigma=1))


## Section 7: INI file alternative

In [ ]:
from ForMoSA.config.global_config import ConfigGenerator
ConfigGenerator().save(str(TUTORIAL_DIR), "config.ini")
print(f"Template: {TUTORIAL_DIR / 'config.ini'}")
print("Edit and load with ConfigLoader — see Tutorial 1 Section 7 for the pattern.")


## Section 8: Next steps

- **Tutorial 5 — Advanced plotting:** Use the results from this (or Tutorial 2)
  to explore every ForMoSA plot in depth. No fitting required.
- **Tutorial 6 — Cluster / MPI deployment:** Scale up to 300+ live points
  using PyMultiNest on an HPC cluster.
